In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_North Campus, DU, Delhi - IMD.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,Ozone,...,Toluene,Eth-Benzene,RH,WS,WD,BP,Xylene,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,137.85,240.18,15.94,29.64,28.73,NaN,0.76,21.46,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,167.31,293.64,25.22,26.38,34.54,NaN,1.07,20.55,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,218.68,402.08,58.86,38.05,68.08,NaN,1.81,18.39,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,267.42,429.89,49.87,55.58,70.12,NaN,1.96,25.47,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,156.21,258.94,17.36,38.06,34.35,NaN,0.83,15.09,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,333.39,499.75,24.63,116.53,49.25,NaN,2.63,105.44,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,282.67,447.25,19.49,121.93,47.38,NaN,2.71,100.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,229.11,392.16,22.16,122.69,49.36,NaN,2.43,93.73,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,240.84,396.11,31.29,124.95,56.04,NaN,2.74,80.18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 12)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 11)
          From Date           To Date   PM2.5     PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  137.85  240.180  15.94  29.64  28.73   
1  02-01-2025 00:00  03-01-2025 00:00  167.31  293.640  25.22  26.38  34.54   
2  03-01-2025 00:00  04-01-2025 00:00   57.74  402.080  12.74  38.05  68.08   
3  04-01-2025 00:00  05-01-2025 00:00   57.74  159.475  49.87  55.58  70.12   
4  05-01-2025 00:00  06-01-2025 00:00  156.21  258.940  17.36  38.06  34.35   

     CO  Ozone  Benzene  TOT-RF  
0  0.76  21.46      0.0       0  
1  1.07  20.55      0.0       0  
2  1.81  18.39      0.0       0  
3  1.96  25.47      0.0       0  
4  0.83  15.09      0.0       0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.061254,0.911796,0.011158,-0.165205,0.121407,-0.674855,-0.917632,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.882472,1.567145,0.756742,-0.375320,0.463314,-0.212108,-0.956156,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.171867,2.896478,-0.245940,0.376840,2.437074,0.892514,-1.047596,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.171867,-0.077542,2.737200,1.506692,2.557124,1.116424,-0.747875,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.573052,1.141769,0.125245,0.377485,0.452132,-0.570364,-1.187297,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.171867,-0.077542,0.709339,-0.029532,1.328967,2.116554,2.637538,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.171867,-0.077542,0.296376,-0.029532,1.218921,2.235973,2.417827,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.171867,2.774872,0.510892,-0.029532,1.335440,1.818008,2.141813,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.171867,2.823294,1.244425,-0.029532,1.728544,2.280755,1.568193,0.0,0.0


In [11]:
df.to_excel('NorthCampus2025.xlsx', index=False)